# Lab: Local OCR (EasyOCR) + RAG on a Scanned PDF (OpenRouter)

## Setup

### Step 1: Install Dependencies


In [ ]:
!pip install pymupdf Pillow sentence-transformers faiss-cpu requests easyocr



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Import Libraries

In [46]:
import os  
import time  
import requests  
import numpy as np  # work with the OCR image arrays  

import fitz  # PyMuPDF -> renders PDF pages to images
from PIL import Image  # store/handle each rendered page as an image

import easyocr  # Local OCR model
from sentence_transformers import SentenceTransformer  # Text to vector model
import faiss  # Vector search database


### Step 3: Setup API Keys

In [47]:
OPENROUTER_API_KEY = input("Please enter your OpenRouter API key manually: ").strip()

# Endpoint + model we'll use later to ask questions about the document
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
TEXT_MODEL = "openai/gpt-oss-20b:free"


### Step 4: Initialize Local AI Models

In [48]:
print("Loading embedding model (for vector search)...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Loading EasyOCR model (for reading text from images)...")
# gpu=False forces CPU mode. Change to True if you have a powerful NVIDIA GPU.
ocr_reader = easyocr.Reader(["en"], gpu=False)

print("Models loaded successfully!")

Loading embedding model (for vector search)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5734.33it/s]
Using CPU. Note: This module is much faster with a GPU.


Loading EasyOCR model (for reading text from images)...
Models loaded successfully!


### Step 5: Download the Scanned PDF

In [49]:
PDF_URL = "https://raw.githubusercontent.com/jamalmazrui/pdf2ocr/master/debate.pdf"
os.makedirs("data", exist_ok=True)  # create a "data" folder if it doesn't exist yet
PDF_PATH = os.path.join("data", "debate.pdf")

# Download the sample scanned PDF and save it locally.
try:
    print("Downloading PDF...")
    r = requests.get(PDF_URL)
    r.raise_for_status()  # stop here if the download failed (e.g. bad URL, no internet)
    
    with open(PDF_PATH, "wb") as f:
        f.write(r.content)
    print(f"Success! Saved PDF to: {PDF_PATH}")
except Exception as e:
    print(f"Error downloading PDF: {e}")


Success! Saved PDF to: data\debate.pdf


### Step 6: Convert PDF Pages to Images

In [50]:
pages = []  # will hold one PIL Image per PDF page

try:
    doc = fitz.open(PDF_PATH)
    zoom = 200 / 72  # Scale up for better image quality (simulating 200 DPI)
    matrix = fitz.Matrix(zoom, zoom)  # transform used when rendering each page

    for page in doc:
        pix = page.get_pixmap(matrix=matrix)  # render the page to raw pixels
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)  # turn those pixels into a normal image object
        pages.append(img)

    doc.close()
    print(f"Success: Converted {len(pages)} pages into images.")
except Exception as e:
    print(f"Error reading PDF: {e}")


Success: Converted 32 pages into images.


### Step 7: Run Vision AI (OCR) to Extract Text

In [51]:
TEST_PAGE_LIMIT = 2  # only process the first 2 pages for this test run
pages_to_process = pages[:TEST_PAGE_LIMIT]
print(f"Running OCR on {len(pages_to_process)} pages...")

chunks = []  # We will store our text chunks here
chunk_size = 450  # how far we move forward for each new chunk (see overlap note below)

for i, page_img in enumerate(pages_to_process, start=1):
    # Convert image to a format EasyOCR understands
    page_array = np.array(page_img)

    # Extract text from the image
    lines = ocr_reader.readtext(page_array, detail=0)
    full_page_text = "\n".join(lines)

    # Split the page text into smaller chunks for our vector database.
    # Each chunk is 500 characters, but we only step forward by 450 -> chunks
    # overlap by 50 characters so we don't accidentally cut a sentence in half
    # right at a chunk boundary.
    for start_idx in range(0, len(full_page_text), chunk_size):
        piece = full_page_text[start_idx : start_idx + 500].strip()
        if piece:
            chunks.append({"page": i, "text": piece})

    print(f"Page {i} transcribed! Found {len(full_page_text)} characters.")

print(f"\nTotal chunks created: {len(chunks)}")


Running OCR on 2 pages...


c:\Users\risha\.virtualenvs\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Page 1 transcribed! Found 1116 characters.
Page 2 transcribed! Found 1283 characters.

Total chunks created: 6


### Step 8: Build the Vector Database (FAISS)

In [52]:
# Get just the text from our chunks
chunk_texts = [c["text"] for c in chunks]

# Convert text to vectors
print("Converting text to vectors...")
embeddings = embed_model.encode(chunk_texts, convert_to_numpy=True)

# Build the FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print(f"Success: Indexed {index.ntotal} chunks into the database.")

Converting text to vectors...
Success: Indexed 6 chunks into the database.


### Step 9: Define the RAG Pipeline

In [53]:
def retrieve_documents(question, k=3):
    """Searches the FAISS database for the most relevant text chunks."""
    try:
        # Convert the question to a vector and search FAISS
        q_vec = embed_model.encode([question], convert_to_numpy=True)
        _, idx = index.search(q_vec, k)
        
        # Grab the actual chunks based on the search results
        retrieved_chunks = [chunks[i] for i in idx[0]]
        return retrieved_chunks
    except Exception as e:
        print(f"Retrieval Error: {e}")
        return []

In [54]:
def execute_rag_pipeline(question):
    """Runs RAG and gets both the answer and source reasoning in a single call."""
    
    sources = retrieve_documents(question)
    if not sources:
        return "No sources found."
        
    # Format context with page numbers clearly labeled
    context_block = "\n\n".join([f"--- Context Block (Page {c['page']}) ---\n{c['text']}" for c in sources])
    
    # This is the instruction ("prompt") we send to the AI model, telling it
    # exactly how to answer and how to format its response.
    qa_prompt = f"""
    You are an expert assistant. Answer the question using ONLY the provided context blocks.
    
    Context:
    {context_block}
    
    Question: {question}
    
    CRITICAL INSTRUCTIONS:
    Output your response in EXACTLY two sections as shown below.
    
    --- FINAL ANSWER ---
    [Provide a direct, 1-sentence answer without bold text or markdown formatting. End with simple citation like (Page X).]
    
    --- EXPLAINABILITY ---
    [For each Context Block provided above, list Page X and state either "NOT USED" or "USED (Extracted: <1 short fact>)"]
    """
    
    payload = {
        "model": TEXT_MODEL,
        "messages": [{"role": "user", "content": qa_prompt}],
        "temperature": 0.0
    }
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
    
    try:
        resp = requests.post(OPENROUTER_URL, headers=headers, json=payload)
        resp.raise_for_status()
        return resp.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        return f"An error occurred: {e}"

### Step 10: Run a Query and See Explainability

In [55]:
user_query = "What was the main topic of the debate?"

print(f"Asking AI: '{user_query}'\n")

# Single call gets both answer and explainability
result = execute_rag_pipeline(user_query)

print(result)

Asking AI: 'What was the main topic of the debate?'

--- FINAL ANSWER ---
The main topic of the debate was foreign policy and homeland security (Page 2).

--- EXPLAINABILITY ---
Page 1: NOT USED  
Page 2: USED (Extracted: The topic of the September 30 debate shall be foreign policy and homeland security)  
Page 1: NOT USED
